# 🏛️ MedStream — مستودع البيانات (Data Warehouse)

أهلاً بيك يا بشمهندس! 👋  
في الملف ده، هنعمل خطوة أساسية ومهمة جداً في هندسة البيانات (Data Engineering):  
**تحويل البيانات النظيفة من مجرد ملف CSV عادي إلى مستودع بيانات حقيقي (Data Warehouse) باستخدام DuckDB.**

---
### 💡 يعني إيه Data Warehouse؟ وليه بنعملها؟
1. **مش مجرد ملف:** في الشركات والمشاريع الحقيقية، الداتا ماينفعش تفضل ملف CSV على الجهاز؛ لأن الـ CSV بطيء وممكن أي حد يعدل فيه بالخطأ.
2. **سرعة رهيبة (OLAP):** مستودع البيانات بيخزن البيانات بنظام الأعمدة (Columnar Storage)، وده بيخلي استعلامات SQL والتحليلات سريعة جداً حتى لو عندك ملايين السجلات.
3. **تغذية كل الأنظمة:** مستودع البيانات هو المصدر الوحيد (Single Source of Truth) اللي هيغذي موديل الـ AI ولوحة التحكم (Dashboard).

--- 
## الخطوة 1: استدعاء المكتبات المطلوبة
**الشرح بالمصري:**  
بنادي مكتبة `duckdb` (مستودع البيانات) ومكتبة `pandas` (لقراءة الجدول).

In [ ]:
import duckdb
import pandas as pd

print("Libraries loaded successfully! 🚀")

--- 
## الخطوة 2: إنشاء والاتصال بمستودع البيانات (Data Warehouse Connection)
**الشرح بالمصري:**  
الأمر ده بينشئ ملف قاعدة بيانات رسمي اسمه `medstream_warehouse.duckdb`.  
الملف ده هو المستودع بتاعنا، ولو مش موجود هيتعمل تلقائياً، ولو موجود هيتصل بيه على طول.

In [ ]:
# إنشاء والاتصال بمستودع البيانات
con = duckdb.connect('medstream_warehouse.duckdb')

print("Connected to MedStream Data Warehouse successfully! 🏛️")

--- 
## الخطوة 3: تحميل وتخزين الداتا النظيفة جوه المستودع كجدول رسمي
**الشرح بالمصري:**  
بنقرا ملف الداتا النظيفة `diabetes_cleaned.csv` ونخزنه بضغطة واحدة كجدول دائم رسمي اسمه `patients_analytics` داخل الـ Data Warehouse.

In [ ]:
# قراءة ملف الداتا النظيفة
df_clean = pd.read_csv('data/processed/diabetes_cleaned.csv')

# تخزين الداتا كجدول تحليلي دائم جوه الـ Warehouse
con.register('temp_clean_data', df_clean)
con.execute("CREATE OR REPLACE TABLE patients_analytics AS SELECT * FROM temp_clean_data;")

print("Table 'patients_analytics' created and loaded successfully! ✅")

--- 
## الخطوة 4: التأكد من عدد السجلات (Row Count Check)
**الشرح بالمصري:**  
بنكتب أمر SQL بسيط (`COUNT(*)`) عشان نتأكد إن الـ 96,128 مريض كلهم اتخزنوا بسلام جوه المستودع.

In [ ]:
# الاستعلام عن إجمالي عدد المرضى في المستودع
total_rows = con.execute("SELECT COUNT(*) FROM patients_analytics").fetchone()[0]
print(f"إجمالي المرضى داخل مستودع البيانات: {total_rows:,} مريض")

--- 
## الخطوة 5: استعلام SQL تحليلي سريع (Data Warehouse Query)
**الشرح بالمصري:**  
ميزة الـ Data Warehouse إننا نقدر نسألها بـ SQL أي سؤال سريري معقد وتجاوبنا في أجزاء من الثانية!  
السؤال هنا: **ما هو متوسط السكر التراكمي وسكر الدم والـ BMI بين المصابين وغير المصابين؟**

In [ ]:
# استعلام تحليلي مجمع (Analytical Aggregation)
query = """
    SELECT 
        CASE WHEN diabetes = 1 THEN 'Diabetic' ELSE 'Non-Diabetic' END AS patient_status,
        COUNT(*) AS total_patients,
        ROUND(AVG(age), 1) AS avg_age,
        ROUND(AVG(blood_glucose_level), 1) AS avg_glucose,
        ROUND(AVG(HbA1c_level), 2) AS avg_hba1c,
        ROUND(AVG(bmi), 1) AS avg_bmi
    FROM patients_analytics
    GROUP BY diabetes;
"""

# عرض النتيجة في جدول أنيق بـ Pandas
result_df = con.execute(query).df()
result_df

--- 
## الخطوة 6: قفل الاتصال بأمان (Close Connection)
**الشرح بالمصري:**  
بعد ما بنخلص الشغل وتخزين الداتا، بنقفل الاتصال عشان كل التغييرات تتحفظ بأمان على الهارد ديسك.

In [ ]:
con.close()
print("Data Warehouse saved and connection closed successfully! 🔒")